# 02 | Structured Excel Automation — Schema Validation & Excel Generator

**LangChain Deep Agents 교육과정 · 실습 2/4 | Google Colab**

**학습 목표**: ① 01번 PDF/이미지 추출 결과 불러오기 ② Pydantic 스키마 검증 ③ 날짜·상태 정규화 및 중복 처리 ④ 업무 유형별 Excel 자동 생성 ⑤ JSON↔Excel 결과 일치 검증 ⑥ LangChain Custom Tool 및 선택적 Deep Agent 통합.

> **실습 구분**: `DEMO-*` 레코드는 모두 가상 데이터입니다. 실제 기업 데이터를 OpenAI API로 자동 전송하지 않습니다. 기본 실행은 **API Key 없이 무료 로컬 처리**이며, Deep Agent 호출은 선택 사항입니다.

**파일 연결**: `01_Multimodal_Document_Extraction.ipynb`의 출력 폴더 `multimodal_document_extraction/output/`의 `pdf_page_extraction.json`, `vision_result.json`을 지원합니다. 01번과 별도 런타임이면 업로드할 수 있습니다.

## 학습 순서 (90~120분)

| 단계 | 작업 | 산출물 |
|---|---|---|
| 1 | 실행 환경·원본 로딩 | 입력 JSON 파악 |
| 2 | 추출 레코드 구성 | `raw_requests.json` |
| 3 | 정규화·Pydantic 검증·중복 판정 | `validation_report.json` |
| 4 | 업무 유형에 따른 Excel 시트 생성 | `MRO_Structured_Excel.xlsx` |
| 5 | Excel 내용 일치성 검증 | `roundtrip_validation.json` |
| 6 | Custom Tool / 선택적 Deep Agent | 호출 기록 및 파일 |

**주의:** JSON 데이터의 형식 검증(`VALID`)과 실제 PDF의 사실 확인(`SOURCE_REVIEW_REQUIRED`)은 별개입니다. Vision 결과가 불명확할 때 검증 완료라고 판단하지 않습니다.

## 1. 개발 환경 준비

Pydantic은 업무 필드를 타입과 제약조건으로 검증합니다. `artifact_tool`은 Excel 워크북을 생성하고 서식을 지정합니다. `langchain-core`의 `@tool`을 사용해 자동화 기능을 Agent용 도구로 등록합니다.

In [ ]:
%pip -q install -U "pydantic>=2" artifact-tool langchain-core langchain-openai deepagents pandas

In [ ]:
from pathlib import Path
from datetime import date, datetime
import os, sys, json, re, shutil
from collections import Counter
from typing import Optional
import pandas as pd
from IPython.display import display

IS_COLAB = 'google.colab' in sys.modules
ROOT = Path('/content/structured_excel_automation') if IS_COLAB else Path.cwd() / 'structured_excel_automation'
INPUT = ROOT / 'input'; OUTPUT = ROOT / 'output'
for d in [ROOT, INPUT, OUTPUT]: d.mkdir(parents=True, exist_ok=True)
PREVIOUS_OUTPUT = Path('/content/multimodal_document_extraction/output') if IS_COLAB else Path.cwd() / 'multimodal_document_extraction/output'
print('Root:', ROOT, '| Previous 01 output exists:', PREVIOUS_OUTPUT.exists())

## 2. 01번 Notebook 결과 가져오기

01번 노트북에서는 `pdf_page_extraction.json`에 페이지별 `text`, `tables`, `page`를 저장하고, OpenAI Vision을 실행했다면 `vision_result.json`에 `extracted` 및 `source_page`를 저장합니다.

동일 Colab 런타임이면 파일을 자동 복사합니다. 별도 런타임에서는 `UPLOAD_PRIOR_FILES=True`로 변경한 뒤 JSON 파일을 직접 업로드하세요. 업로드하지 않아도 내장 교육용 레코드로 실습할 수 있습니다.

In [ ]:
for name in ['pdf_page_extraction.json', 'vision_result.json']:
    src = PREVIOUS_OUTPUT / name
    if src.is_file():
        shutil.copy2(src, INPUT / name)
        print('Imported:', name)
UPLOAD_PRIOR_FILES = False
if UPLOAD_PRIOR_FILES and IS_COLAB:
    from google.colab import files
    for name, data in files.upload().items():
        if name not in {'pdf_page_extraction.json', 'vision_result.json'}:
            print('Ignored unknown filename:', name); continue
        (INPUT / name).write_bytes(data)
print('Available:', [p.name for p in INPUT.glob('*.json')])

### 2-1. 안전한 JSON 로드 및 원본 구조 확인

입력 파일을 임의 코드로 실행하지 않고 JSON으로 파싱합니다. `vision_result.json`이 없는 것은 01번에서 API 실행을 꺼두었다는 뜻일 수 있으므로 정상입니다.

In [ ]:
def read_json(path: Path, default):
    if not path.is_file(): return default
    try: return json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError as exc: raise ValueError(f'잘못된 JSON: {path.name}: {exc}')

pages = read_json(INPUT/'pdf_page_extraction.json', [])
vision_blob = read_json(INPUT/'vision_result.json', None)
print('PDF page records:', len(pages))
print('Vision record:', bool(vision_blob))
if pages: print('Sample keys:', list(pages[0]))
if vision_blob: print('Vision keys:', list(vision_blob))

## 3. 원본 데이터에서 업무 레코드 만들기

문서 1페이지에는 `Receipt ID:`, `Received date:`, `Customer:`, `System:`, `Request:`, `Status:` 같은 라벨이 있습니다. **텍스트 추출만 가능한 페이지**를 위한 결정적 파서를 작성합니다. 01번 스캔 페이지는 텍스트가 없으므로 Vision 결과가 있을 때만 포함합니다.

페이지별 추출 항목에 `source_file`, `source_page`, `source_method`를 연결하여 출처를 잃지 않도록 합니다.

In [ ]:
LABEL_TO_KEY = {
    'Receipt ID': 'receipt_id', 'Received date': 'received_date',
    'Customer': 'customer', 'System': 'system_name',
    'Request': 'request_description', 'Status': 'processing_status',
}
def parse_labelled_page(text: str) -> dict:
    out = {}
    for line in str(text).splitlines():
        for label, key in LABEL_TO_KEY.items():
            if line.lower().startswith(label.lower()+':'):
                out[key] = line.split(':', 1)[1].strip()
    return out

raw_records = []
for p in pages:
    parsed = parse_labelled_page(p.get('text',''))
    if parsed:
        raw_records.append({**parsed,
            'source_file': 'demo_mro_requests.pdf',
            'source_page': p.get('page'), 'source_method': 'PDF_TEXT',
            'uncertain_fields': []})
if isinstance(vision_blob, dict) and isinstance(vision_blob.get('extracted'), dict):
    raw_records.append({**vision_blob['extracted'],
        'source_file': vision_blob.get('source_file', 'unknown.pdf'),
        'source_page': vision_blob.get('source_page'),
        'source_method': 'VISION',
        'uncertain_fields': vision_blob['extracted'].get('uncertain_fields',[])})
print('Records from 01:', len(raw_records))

### 3-1. 독립 실행용 샘플 추가 (반드시 가상 데이터)

이전 노트북을 실행하지 않은 수강생도 실습할 수 있도록 **가상 데이터**를 자동 추가합니다. 일치 사례, 날짜 형식 차이, 필수값 누락, 중복 접수번호를 일부러 넣어 검증 로직을 체험합니다.

이 레코드들은 실제 01번 추출 결과라고 주장하지 않으며, `source_method=TRAINING_SAMPLE`로 구별합니다.

In [ ]:
DEMO_RECORDS = [
 {'receipt_id':'DEMO-2026-076','received_date':'2026/07/22','customer':'Demo Manufacturing','system_name':'MRO Portal','request_description':'Send automatic email two days before service date','processing_status':'Deployment approved'},
 {'receipt_id':'DEMO-2026-072','received_date':'2026-07-23','customer':'Demo Services','system_name':'ERP','request_description':'Fix invoice item name mapping','processing_status':'Received'},
 {'receipt_id':'DEMO-2026-073','received_date':'2026-07-24','customer':'Sample Client','system_name':'ERP','request_description':'Improve employee evaluation screen','processing_status':'Completed'},
 {'receipt_id':'DEMO-2026-074','received_date':'2026-13-05','customer':'Sample Client','system_name':'RPA','request_description':'Create monthly report','processing_status':'In progress'},
 {'receipt_id':'DEMO-2026-075','received_date':'2026-07-26','customer':'Sample Client','system_name':'ERP','request_description':'','processing_status':'Received'},
 {'receipt_id':'DEMO-2026-072','received_date':'2026-07-23','customer':'Demo Services','system_name':'ERP','request_description':'Fix invoice item name mapping','processing_status':'Received'},
]
for r in DEMO_RECORDS:
    raw_records.append({**r, 'source_file':'TRAINING_SAMPLE', 'source_page':None,
                        'source_method':'TRAINING_SAMPLE','uncertain_fields':[]})
(RAW_PATH := OUTPUT/'raw_requests.json').write_text(json.dumps(raw_records,ensure_ascii=False,indent=2),encoding='utf-8')
print('Combined records:',len(raw_records),'| Saved:', RAW_PATH)
display(pd.DataFrame(raw_records)[['receipt_id','received_date','processing_status','source_method']])

## 4. 업무별 표준 스키마 정의 (Pydantic)

접수번호는 `DEMO-YYYY-NNN` 형식으로 검증합니다. 이는 교육용 규칙이므로 실제 운영에서는 회사의 접수번호 규칙으로 변경해야 합니다.

처리상태는 표준 4종(접수, 진행중, 처리완료, 배포승인)에 한정하며, 원문 `Received`, `Completed` 등을 명시적 맵으로 정규화합니다. 매핑되지 않는 값은 자동 추정하지 않습니다.

In [ ]:
from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Literal

STATUS_MAP = {
 'received':'접수','접수':'접수',
 'in progress':'진행중','진행중':'진행중',
 'completed':'처리완료','처리완료':'처리완료',
 'deployment approved':'배포승인','배포승인':'배포승인'
}
STATUS_VALUES = Literal['접수','진행중','처리완료','배포승인']

class MRORequest(BaseModel):
    receipt_id: str = Field(min_length=1)
    received_date: date
    customer: str = Field(min_length=1)
    system_name: str = Field(min_length=1)
    request_description: str = Field(min_length=1)
    processing_status: STATUS_VALUES
    source_file: str
    source_page: Optional[int] = None
    source_method: str
    uncertain_fields: list[str] = Field(default_factory=list)

    @field_validator('receipt_id')
    @classmethod
    def validate_receipt_id(cls, value):
        if not re.fullmatch(r'DEMO-\d{4}-\d{3}', value):
            raise ValueError('교육용 접수번호 형식 DEMO-YYYY-NNN 위반')
        return value

    @field_validator('customer','system_name','request_description')
    @classmethod
    def reject_blank(cls, value):
        if not value or not value.strip(): raise ValueError('빈 값 불허')
        return value.strip()

print('Fields:', list(MRORequest.model_fields))

### 4-1. 정규화와 검증은 분리

날짜의 `/`는 `-`로 변환할 수 있지만 `2026-13-05`처럼 유효하지 않은 날짜는 자동 보정하면 안 됩니다. `None` 또는 문서에 없는 필드는 그대로 남겨 에러로 기록합니다.

In [ ]:
def normalize_record(raw: dict) -> dict:
    cleaned = dict(raw)
    rid = cleaned.get('receipt_id')
    cleaned['receipt_id'] = str(rid).strip() if rid is not None else ''
    dt = cleaned.get('received_date')
    cleaned['received_date'] = str(dt).strip().replace('/','-') if dt is not None else None
    status = cleaned.get('processing_status')
    key = str(status).strip().lower() if status is not None else ''
    cleaned['processing_status'] = STATUS_MAP.get(key, status)
    for field in ('customer','system_name','request_description'):
        v = cleaned.get(field)
        cleaned[field] = str(v).strip() if v is not None else ''
    cleaned.setdefault('uncertain_fields', [])
    return cleaned

# 상태가 명시적 사전에 있는지 확인
for x in ('Received', 'Completed', 'Deployment approved', 'unknown'):
    print(x, '=>', normalize_record({'processing_status': x})['processing_status'])

## 5. 검증, 중복 탐지 및 검토 대기 분리

검증 실패 또는 동일 접수번호 반복은 `REVIEW_REQUIRED`로 분리합니다. **중복 접수번호는 무조건 삭제하지 않습니다.** 실제 MRO 데이터에서는 같은 접수번호의 하위작업일 수 있으므로 원본 행을 보존하고 처리 방법을 확인해야 합니다.

또한 Vision의 `uncertain_fields`가 있으면 형식 검증에 통과해도 담당자 검토 대상으로 둡니다.

In [ ]:
def validate_requests(records: list[dict]) -> tuple[list[dict], list[dict]]:
    valid, review = [], []
    counts = Counter(str(x.get('receipt_id') or '').strip() for x in records)
    for row_idx, raw in enumerate(records, start=1):
        cleaned = normalize_record(raw)
        issues = []
        try:
            parsed = MRORequest.model_validate(cleaned)
            record = parsed.model_dump(mode='json')
        except ValidationError as e:
            record = cleaned
            issues.extend(f"{'.'.join(map(str,x['loc']))}: {x['msg']}" for x in e.errors())
        if counts[cleaned['receipt_id']] > 1:
            issues.append('DUPLICATE_RECEIPT_ID: 하위작업 여부 확인 필요')
        if cleaned.get('uncertain_fields'):
            issues.append('UNCERTAIN_VISION_FIELDS: 사람의 원본 확인 필요')
        record['source_row'] = row_idx
        if issues:
            review.append({**record, 'validation_status':'REVIEW_REQUIRED', 'issues':'; '.join(issues)})
        else:
            valid.append({**record, 'validation_status':'VALID', 'issues':''})
    return valid, review

valid_records, review_records = validate_requests(raw_records)
print('VALID:',len(valid_records),'| REVIEW_REQUIRED:',len(review_records))
display(pd.DataFrame([{'receipt_id':r.get('receipt_id'), 'issues':r['issues']} for r in review_records]))

### 5-1. 검증 보고서 파일 저장

이 파일은 다음 단계(03번 System Query & Reconciliation)의 입력으로 재사용할 수 있습니다. 원본 추출 건수 = VALID + REVIEW_REQUIRED가 일치하는지도 검사합니다.

In [ ]:
validation_report = {
 'total':len(raw_records),'valid_count':len(valid_records),'review_count':len(review_records),
 'valid':valid_records,'review_required':review_records,
 'policy': 'Schema VALID does not imply source verified'
}
assert validation_report['total'] == validation_report['valid_count'] + validation_report['review_count']
(OUTPUT/'validation_report.json').write_text(json.dumps(validation_report,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT/'validated_requests.json').write_text(json.dumps(valid_records,ensure_ascii=False,indent=2),encoding='utf-8')
print('Saved validation_report.json and validated_requests.json')

## 6. Excel 생성 설계

Excel은 4개 시트로 만듭니다.

| 시트 | 역할 |
|---|---|
| `요구사항_검증완료` | 스키마 검증을 통과한 데이터 |
| `검토필요` | 형식 오류·중복·불확실 Vision 레코드 및 사유 |
| `업무유형별_요약` | 시스템별·처리상태별 건수와 품질 지표 |
| `데이터_출처` | 원본 파일, 페이지, 추출 방법, 원본 행 번호 |

**Excel 생성 도구:** `artifact_tool`로 시트, 헤더 서식, 열 폭, 표 및 요약을 구성합니다. 구조화된 레코드를 저장하되 실행 코드나 수식을 원본 텍스트로부터 만들지 않습니다.

In [ ]:
from artifact_tool import Workbook, SpreadsheetFile

DATA_COLS = [
 ('receipt_id','접수번호'),('received_date','접수일자'),('customer','고객사'),
 ('system_name','업무시스템'),('request_description','요구사항'),
 ('processing_status','처리상태'),('source_file','원본파일'),
 ('source_page','원본페이지'),('source_method','추출방법'),('source_row','원본행번호')
]
REVIEW_COLS = DATA_COLS + [('issues','검토사유')]

def excel_safe(value):
    if value is None: return ''
    if isinstance(value, (datetime,date)): return value.isoformat()
    if isinstance(value, (list,dict)): return json.dumps(value,ensure_ascii=False)
    value = str(value) if not isinstance(value, (int,float,bool)) else value
    # spreadsheet formula injection 방지: 수식처럼 해석될 수 있는 입력을 문자값으로 보호
    if isinstance(value,str) and value.lstrip().startswith(('=','+','-','@')):
        return "'"+value
    return value

def matrix(rows, columns):
    return [[excel_safe(r.get(key)) for key,_ in columns] for r in rows]

HEADER_FORMAT = {
 'fill':'#12304A',
 'font':{'bold':True,'color':'#FFFFFF'},
 'horizontal_alignment':'center',
 'vertical_alignment':'center',
 'row_height':28,
}

def add_data_sheet(wb, name, columns, records, table_name):
    sheet = wb.worksheets.add(name)
    headers = [label for _,label in columns]
    values = [headers] + matrix(records, columns)
    sheet.get_range_by_indexes(0,0,len(values),len(columns)).values = values
    sheet.get_range_by_indexes(0,0,1,len(columns)).format = HEADER_FORMAT
    sheet.freeze_panes.freeze_rows(1)
    for j,(key,label) in enumerate(columns):
        col_range = sheet.get_range_by_indexes(0,j,max(2,len(values)),1)
        col_range.format.column_width = 34 if key in {'request_description','issues'} else (24 if key in {'source_file','customer'} else 18)
    if records:
        data_range = sheet.get_range_by_indexes(1,0,len(records),len(columns))
        data_range.format.wrap_text = True
        sheet.tables.add(sheet.get_range_by_indexes(0,0,len(values),len(columns)),True,table_name)
    return sheet

### 6-1. 워크북 생성 및 시각적 구성

업무용 Excel의 출처를 한눈에 확인할 수 있도록 출처 시트를 별도로 저장합니다. 요약표는 검증 상태와 원본 행 수를 포함하며, 이 수치는 Python의 실제 검증 결과에서 계산합니다.

In [ ]:
wb = Workbook.create()
add_data_sheet(wb,'요구사항_검증완료',DATA_COLS,valid_records,'ValidatedRequests')
add_data_sheet(wb,'검토필요',REVIEW_COLS,review_records,'ReviewRequired')

source_records = [
 {'receipt_id':r.get('receipt_id'),'source_file':r.get('source_file'),
  'source_page':r.get('source_page'),'source_method':r.get('source_method'),
  'source_row':i, 'validation_status':'REVIEW_REQUIRED' if any(x['source_row']==i for x in review_records) else 'VALID'}
 for i,r in enumerate(raw_records, start=1)
]
SOURCE_COLS=[('receipt_id','접수번호'),('source_file','원본파일'),('source_page','원본페이지'),('source_method','추출방법'),('source_row','원본행번호'),('validation_status','검증상태')]
add_data_sheet(wb,'데이터_출처',SOURCE_COLS,source_records,'SourceTrace')

summary = wb.worksheets.add('업무유형별_요약')
summary.get_range('A1:D1').values = [['MRO 요구사항 처리 현황','','','']]
summary.merge_cells('A1:D1')
summary.get_range('A1:D1').format = HEADER_FORMAT
summary.get_range('A3:B6').values = [
 ['지표','건수'],['전체 원본 레코드',len(raw_records)],
 ['VALID',len(valid_records)],['REVIEW_REQUIRED',len(review_records)]
]
summary.get_range('A3:B3').format = HEADER_FORMAT
status_counts = Counter(r['processing_status'] for r in valid_records)
system_counts = Counter(r['system_name'] for r in valid_records)
summary.get_range('D3:E3').values = [['처리상태(VALID)','건수']]
summary.get_range('D3:E3').format = HEADER_FORMAT
if status_counts:
    summary.get_range_by_indexes(3,3,len(status_counts),2).values = [[k,v] for k,v in sorted(status_counts.items())]
summary.get_range('G3:H3').values = [['업무시스템(VALID)','건수']]
summary.get_range('G3:H3').format = HEADER_FORMAT
if system_counts:
    summary.get_range_by_indexes(3,6,len(system_counts),2).values = [[k,v] for k,v in sorted(system_counts.items())]
for col in 'ABCDEFGH': summary.get_range(f'{col}:{col}').format.column_width = 22
summary.get_range('A8:D8').values = [['* VALID는 스키마 검증 통과이며 원본 사실 검증 완료를 의미하지 않습니다.','','','']]
summary.merge_cells('A8:D8')

EXCEL_PATH = OUTPUT/'MRO_Structured_Excel.xlsx'
SpreadsheetFile.export_xlsx(wb).save(str(EXCEL_PATH))
print('Excel 생성 완료:',EXCEL_PATH,'| bytes:',EXCEL_PATH.stat().st_size)

## 7. 산출물 일치 검증 (Round-trip)

Excel 저장 이후 파일을 다시 읽어 검증완료 및 검토필요 시트의 접수번호와 건수를 원본 Python 결과와 비교합니다. 이 검증은 **데이터 보존 검사**이며 Vision 인식 정답률이나 ERP와의 대사 검증은 아닙니다.

In [ ]:
from artifact_tool import Blob
reloaded = SpreadsheetFile.import_xlsx(Blob.load(str(EXCEL_PATH)))

def read_saved_rows(sheet_name: str, columns: list[tuple[str,str]], count: int):
    if count==0: return []
    sh = reloaded.worksheets.get_item(sheet_name)
    vals = sh.get_range_by_indexes(1,0,count,len(columns)).values
    return [dict(zip([k for k,_ in columns],row)) for row in vals]

saved_valid = read_saved_rows('요구사항_검증완료',DATA_COLS,len(valid_records))
saved_review = read_saved_rows('검토필요',REVIEW_COLS,len(review_records))
expected_valid_ids = [str(r.get('receipt_id')) for r in valid_records]
expected_review_ids = [str(r.get('receipt_id')) for r in review_records]
check_results = {
 'excel_exists':EXCEL_PATH.is_file(),
 'valid_count_match':len(saved_valid)==len(valid_records),
 'review_count_match':len(saved_review)==len(review_records),
 'valid_receipt_order_match':[str(r['receipt_id']) for r in saved_valid]==expected_valid_ids,
 'review_receipt_order_match':[str(r['receipt_id']) for r in saved_review]==expected_review_ids,
 'input_partition_match':len(raw_records)==len(valid_records)+len(review_records),
}
(OUTPUT/'roundtrip_validation.json').write_text(json.dumps(check_results,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(check_results,ensure_ascii=False,indent=2))
assert all(check_results.values()), '저장된 Excel 내용 검증 실패'
print('PASS: 행 수 및 접수번호 보존')

## 8. LangChain Custom Tools: 검증과 Excel 생성

Agent에 연결하는 Custom Tool은 임의 경로 접근을 허용하지 않고, 실습 폴더의 검증 데이터만 사용합니다. 실행 기능은 Python에서 확인한 함수만 노출하며 외부 파일을 자동 삭제·변경하지 않습니다.

In [ ]:
from langchain_core.tools import tool

@tool
def get_validation_summary() -> str:
    """MRO 추출 레코드의 스키마 검증 결과 및 검토 사유를 조회한다."""
    report = read_json(OUTPUT/'validation_report.json', {})
    return json.dumps({
        'total':report.get('total',0),
        'valid_count':report.get('valid_count',0),
        'review_count':report.get('review_count',0),
        'review_items':[{'receipt_id':r.get('receipt_id'),'issues':r.get('issues')}
                        for r in report.get('review_required',[])]
    },ensure_ascii=False)

@tool
def get_generated_excel_status() -> str:
    """생성된 MRO Excel 파일의 저장 경로와 Round-trip 검증 결과를 확인한다."""
    checks = read_json(OUTPUT/'roundtrip_validation.json',{})
    return json.dumps({'file':str(EXCEL_PATH),'exists':EXCEL_PATH.is_file(),
                       'validation':checks},ensure_ascii=False)

print(get_validation_summary.invoke({})[:700])
print(get_generated_excel_status.invoke({})[:350])

## 9. OpenAI Deep Agent 연결 (선택 · 유료 API)

기본값 `ENABLE_OPENAI=False`에서는 실제 API를 호출하지 않습니다. True로 변경하면 **Tool의 요약 결과가 OpenAI API로 전달**될 수 있습니다. 회사 데이터는 전송 승인과 비식별 처리가 완료된 경우에만 사용하세요.

여기서 Agent는 검증 보고서를 **읽고 설명**합니다. 구조화 검증과 Excel 생성·검증은 이미 결정적 Python 코드로 실행했습니다.

In [ ]:
ENABLE_OPENAI = False
MODEL_ID = 'gpt-4.1-mini'  # 접근 가능한 Tool Calling 모델로 조정
if ENABLE_OPENAI:
    try:
        from google.colab import userdata
        key = userdata.get('OPENAI_API_KEY')
        if key: os.environ['OPENAI_API_KEY'] = key
    except ImportError:
        pass
    if not os.environ.get('OPENAI_API_KEY'):
        raise RuntimeError('Colab Secrets 또는 환경변수에 OPENAI_API_KEY를 설정하세요.')
else:
    print('OpenAI API 호출 비활성화: Tool 단독 검증과 Excel 생성은 완료되었습니다.')

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage
agent_result = None
if ENABLE_OPENAI:
    from deepagents import create_deep_agent
    agent = create_deep_agent(
        model='openai:'+MODEL_ID,
        tools=[get_validation_summary, get_generated_excel_status],
        system_prompt=(
            '당신은 MRO 정형 데이터 검증 조교입니다. 반드시 Tool 결과만 근거로 ' 
            '검증된 데이터 건수, 검토 사유, Excel 파일 상태를 보고하세요. ' 
            '스키마 VALID를 원본 사실 검증 완료라고 표현하지 마세요.'
        )
    )
    agent_result = agent.invoke({'messages':[{'role':'user','content':
        '검증된 요구사항 건수, 검토 사유, Excel 저장 및 일치 검증 결과를 정리해 주세요.'}]})
    print(agent_result['messages'][-1].content)
else:
    print('Deep Agent 실습은 선택 사항입니다.')

### 9-1. Agent Tool Calling 기록 확인

실제 도구를 호출했는지는 응답 문장이 아니라 Tool Trace로 확인해야 합니다.

In [ ]:
if agent_result is not None:
    for message in agent_result['messages']:
        if isinstance(message, AIMessage) and message.tool_calls:
            for call in message.tool_calls:
                print('CALL',call['name'],call['args'])
        if isinstance(message, ToolMessage):
            print('RESULT',message.name, str(message.content)[:500])

## 10. 다운로드 및 다음 실습 연결

최종 Excel과 검증 보고서(JSON)를 ZIP으로 묶습니다. Colab에서 **파일 다운로드 옵션**을 선택하면 로컬 PC로 저장할 수 있습니다.

다음 `03_System_Query_Reconciliation.ipynb`에서는 검증완료 데이터(`validated_requests.json`)와 ERP Mock 데이터의 접수번호·처리상태·요청 내용 등을 대사하게 됩니다.

In [ ]:
ZIP_FILE = shutil.make_archive(str(ROOT/'02_Structured_Excel_Outputs'),'zip',root_dir=OUTPUT)
print('Excel:',EXCEL_PATH)
print('ZIP:',ZIP_FILE)
DOWNLOAD_NOW = False  # Colab에서 True로 변경하면 ZIP 다운로드
if DOWNLOAD_NOW and IS_COLAB:
    from google.colab import files
    files.download(ZIP_FILE)

## 11. 수강생 과제 및 확인 문제

**과제 A:** `processing_status` 매핑 사전에 `반려`를 추가하고 Pydantic `Literal`에 반영해 보세요.

**과제 B:** 처리완료와 배포승인 건수를 각각 집계해 `업무유형별_요약`에 추가해 보세요.

**과제 C:** 같은 접수번호라도 `subtask_id`가 다르면 중복이 아닌 복수 하위작업으로 식별하도록 검증 정책을 변경해 보세요.

**확인 문제**: (1) Pydantic 검증과 원본 사실 검증은 왜 다른가요? (2) 누락값을 자동 추정하면 어떤 문제가 있나요? (3) 원본 데이터 대사 전에 Excel 저장 후 재검사가 필요한 이유는 무엇인가요?

**다음 단계:** `03_System_Query_Reconciliation.ipynb` — 업무 시스템 조회, 원본·추출·처리결과 대사 자동화.